# 01.07_processing_2021Sponges

整理 Spongilla 数据。

- 当前文件：`analysis/01_preprocessing/01.07_processing_2021Sponges.ipynb`
- 原始来源：`Codes/01.07_processing_2021Sponges.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`pandas`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


# Sponges

In [ ]:
import scanpy as sc
import pandas as pd

# 正确读取并转置表达矩阵
adata = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2021Sponges/Spongilla.h5ad")
adata

In [ ]:
adata.var

In [ ]:
# 备份原始基因名
adata.var['original_gene_names'] = adata.var.index.copy()

# 清洗基因名：只保留第一个空格前的内容
new_index = [gene_name.split(' ')[0] for gene_name in adata.var.index]
adata.var.index = new_index

# 检查结果
print(adata.var.head())

In [ ]:
adata

In [ ]:
# 查找重复的基因名
duplicates = adata.var_names[adata.var_names.duplicated()]
# 输出重复的基因名
print("重复的基因名：", duplicates)
# 删除重复的基因（保留第一个出现的）
adata = adata[:, ~adata.var_names.duplicated()]
adata.var

In [ ]:
# 基因重命名，更换_为-
adata.var_names = adata.var_names.str.replace('_', '-', regex=False)
adata.var

In [ ]:
# 细胞名
adata.obs

In [ ]:
# 矩阵内容
# Values were log1p-normalized, mean-centered and scaled, and filtered for highly variable genes using the same procedure described above for downstream dimensionality reduction and visualization (e.g., PCA) using Scanpy. 
adata.X

In [ ]:
max(adata.X[0])

In [ ]:
adata

In [ ]:
# 标准化和 log 转换
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 高度变异基因选择（如果需要）
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

# 使用 Seurat v1 的类似方法（基于均值和离散度）
# sc.pp.highly_variable_genes(
#     adata,
#     flavor="seurat",  # 最接近 Macosko 2015 的方法
#     n_top_genes=None,  # 不固定数量，用阈值筛选
#     min_mean=0.025,    # x.low.cutoff=.025
#     min_disp=1.0,      # y.cutoff=1
#     max_mean=10,
#     min_disp=0.5
# )
print(adata)

In [ ]:
print(adata.X)
max(adata.X[0])

In [ ]:
# 保存原始数据
adata.raw = adata.copy()

In [ ]:
# 取高可变基因
adata = adata[:, adata.var.highly_variable]

# 使用高变基因进行 PCA
sc.tl.pca(
    adata,
    n_comps=100,       # pcs.compute=100
    svd_solver="arpack",
    # use_highly_variable=True
)

# 可选：查看 PCA 贡献（类似 Seurat 的 ElbowPlot）
sc.pl.pca_variance_ratio(adata, n_pcs=50)

In [ ]:
# 计算邻居图
sc.pp.neighbors(adata, n_neighbors=40, n_pcs=40)

sc.tl.louvain(
    adata,
    resolution=1.0,      # 明确指定分辨率=1
    random_state=42,     # 可复现性
    key_added='louvain'  # 聚类结果存储到 adata.obs['louvain']
)

# 计算 UMAP
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color="louvain", legend_loc="on data", frameon=False)

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['MetaCells', 'CellType'])

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['CellType'])

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['CellTypeAbbreviation', 'CellTypeFamily'])

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['CellTypeFamily'])

In [ ]:
print(len(adata.obs["CellType"].value_counts()))
adata.obs["CellType"].value_counts()

In [ ]:
print(len(adata.obs["CellTypeFamily"].value_counts()))
adata.obs["CellTypeFamily"].value_counts()

In [ ]:
adata

In [ ]:
adata = adata.raw.to_adata()
adata

In [ ]:
# 保存adata数据
raw_spla_path = "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Spla.normalized.h5ad"
adata.write(raw_spla_path)

In [ ]:
# 导出基因id
adata.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Spla.genes.txt', index=False, header=False)